# Fake News Detection - Standalone Environment
Run the cells below sequentially. The first few cells will automatically generate the project structure inside your Colab environment.

In [ ]:
!mkdir -p src
%%writefile src/utils.py
import yaml
import torch

def load_config(config_path):
    with open(config_path, 'r') as f:
        return yaml.safe_load(f)

def get_device(device_str):
    if device_str == "auto":
        if torch.cuda.is_available():
            return torch.device('cuda')
        elif torch.backends.mps.is_available():
            return torch.device('mps')
        else:
            return torch.device('cpu')
    return torch.device(device_str)


In [ ]:
!mkdir -p src
%%writefile src/data.py
import os
import os.path as osp
from torch_geometric.datasets import UPFD
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToUndirected

def get_loaders(config):
    """
    Creates train, val, and test data loaders based on the provided configuration.
    """
    data_cfg = config['data']
    path = osp.join(osp.dirname(osp.realpath(__file__)), '..', data_cfg['data_dir'])
    
    # Common transform to ensure graphs are undirected
    transform = ToUndirected()

    # Use the standard UPFD dataset, which will automatically download if missing
    train_dataset = UPFD(path, data_cfg['dataset'], data_cfg['feature'], 'train', transform)
    val_dataset = UPFD(path, data_cfg['dataset'], data_cfg['feature'], 'val', transform)
    test_dataset = UPFD(path, data_cfg['dataset'], data_cfg['feature'], 'test', transform)

    train_loader = DataLoader(train_dataset, batch_size=data_cfg['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=data_cfg['batch_size'], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=data_cfg['batch_size'], shuffle=False)

    return train_loader, val_loader, test_loader, train_dataset


In [ ]:
!mkdir -p src
%%writefile src/trainer.py
import torch
import torch.nn.functional as F
from tqdm import tqdm

class Trainer:
    def __init__(self, model, optimizer, device, train_loader, val_loader, test_loader):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.device = device
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader

    def _get_model_kwargs(self, data):
        kwargs = {
            'x': data.x,
            'edge_index': data.edge_index,
            'batch': data.batch
        }
        if hasattr(data, 'sentiment'):
            kwargs['sentiment_features'] = data.sentiment
        return kwargs

    def train_epoch(self):
        self.model.train()
        total_loss = 0
        for data in self.train_loader:
            data = data.to(self.device)
            self.optimizer.zero_grad()
            out = self.model(**self._get_model_kwargs(data))
            loss = F.nll_loss(out, data.y)
            loss.backward()
            self.optimizer.step()
            total_loss += float(loss) * data.num_graphs
        return total_loss / len(self.train_loader.dataset)

    @torch.no_grad()
    def test(self, loader):
        self.model.eval()
        total_correct = total_examples = 0
        for data in loader:
            data = data.to(self.device)
            out = self.model(**self._get_model_kwargs(data))
            pred = out.argmax(dim=-1)
            total_correct += int((pred == data.y).sum())
            total_examples += data.num_graphs
        return total_correct / total_examples

    def fit(self, epochs):
        for epoch in range(1, epochs + 1):
            loss = self.train_epoch()
            train_acc = self.test(self.train_loader)
            val_acc = self.test(self.val_loader)
            test_acc = self.test(self.test_loader)
            print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}, Train: {train_acc:.4f}, '
                  f'Val: {val_acc:.4f}, Test: {test_acc:.4f}')

class TransductiveTrainer:
    def __init__(self, model, optimizer, device, giant_batch, H, train_idx, val_idx, test_idx, y):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.device = device
        self.giant_batch = giant_batch.to(device)
        self.H = H.to(device)
        self.train_idx = train_idx.to(device)
        self.val_idx = val_idx.to(device)
        self.test_idx = test_idx.to(device)
        self.y = y.to(device)

    def train_epoch(self):
        self.model.train()
        self.optimizer.zero_grad()
        # Forward pass on the entire hypergraph
        out = self.model(
            x=self.giant_batch.x,
            edge_index=self.giant_batch.edge_index,
            batch=self.giant_batch.batch,
            H=self.H
        )
        # Compute loss on training nodes only
        loss = F.nll_loss(out[self.train_idx], self.y[self.train_idx])
        loss.backward()
        self.optimizer.step()
        return float(loss)

    @torch.no_grad()
    def test(self, indices):
        self.model.eval()
        out = self.model(
            x=self.giant_batch.x,
            edge_index=self.giant_batch.edge_index,
            batch=self.giant_batch.batch,
            H=self.H
        )
        pred = out[indices].argmax(dim=-1)
        correct = int((pred == self.y[indices]).sum())
        return correct / len(indices)

    def fit(self, epochs):
        for epoch in range(1, epochs + 1):
            loss = self.train_epoch()
            train_acc = self.test(self.train_idx)
            val_acc = self.test(self.val_idx)
            test_acc = self.test(self.test_idx)
            print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}, Train: {train_acc:.4f}, '
                  f'Val: {val_acc:.4f}, Test: {test_acc:.4f}')




In [ ]:
!mkdir -p src/models
%%writefile src/models/cmcg.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadCoAttentionLayer(nn.Module):
    def __init__(self, hidden_channels1, hidden_channels2, num_heads=2, dropout_rate=0.01):
        super(MultiHeadCoAttentionLayer, self).__init__()
        self.num_heads = num_heads
        self.hidden_channels1 = hidden_channels1
        self.hidden_channels2 = hidden_channels2

        self.query1 = nn.Linear(hidden_channels1, hidden_channels1 * num_heads, bias=False)
        self.key1 = nn.Linear(hidden_channels1, hidden_channels1 * num_heads, bias=False)
        self.value1 = nn.Linear(hidden_channels1, hidden_channels1 * num_heads, bias=False)

        self.query2 = nn.Linear(hidden_channels2, hidden_channels2 * num_heads, bias=False)
        self.key2 = nn.Linear(hidden_channels2, hidden_channels2 * num_heads, bias=False)
        self.value2 = nn.Linear(hidden_channels2, hidden_channels2 * num_heads, bias=False)

        self.out1 = nn.Linear(hidden_channels1 * num_heads, hidden_channels1)
        self.out2 = nn.Linear(hidden_channels2 * num_heads, hidden_channels2)
        
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x1, x2):
        Q1 = self.query1(x1).view(-1, self.num_heads, self.hidden_channels1)
        K2 = self.key2(x2).view(-1, self.num_heads, self.hidden_channels2)
        V2 = self.value2(x2).view(-1, self.num_heads, self.hidden_channels2)

        Q2 = self.query2(x2).view(-1, self.num_heads, self.hidden_channels2)
        K1 = self.key1(x1).view(-1, self.num_heads, self.hidden_channels1)
        V1 = self.value1(x1).view(-1, self.num_heads, self.hidden_channels1)

        attn_scores1 = torch.matmul(Q1, K2.transpose(-2, -1)) / (self.hidden_channels2 ** 0.5)
        attn_weights1 = self.dropout(F.softmax(attn_scores1, dim=-1))
        attended_x1 = torch.matmul(attn_weights1, V2)

        attn_scores2 = torch.matmul(Q2, K1.transpose(-2, -1)) / (self.hidden_channels1 ** 0.5)
        attn_weights2 = self.dropout(F.softmax(attn_scores2, dim=-1))
        attended_x2 = torch.matmul(attn_weights2, V1)

        attended_x1 = attended_x1.view(-1, self.num_heads * self.hidden_channels1)
        attended_x2 = attended_x2.view(-1, self.num_heads * self.hidden_channels2)

        out_x1 = self.out1(attended_x1) + x1
        out_x2 = self.out2(attended_x2) + x2

        return out_x1, out_x2


In [ ]:
!mkdir -p src/models
%%writefile src/models/classifier.py
import torch.nn as nn
from torch.nn import Linear

class MLPClassifier(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # A simple MLP head. This can be made more complex if needed.
        self.lin = Linear(in_channels, out_channels)

    def forward(self, x):
        h = self.lin(x)
        return h.log_softmax(dim=-1)


In [ ]:
!mkdir -p src/models
%%writefile src/models/text_encoders.py
import torch
from torch.nn import Linear

class TextEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.lin = Linear(in_channels, hidden_channels)

    def forward(self, x, batch):
        # Get the root node (news content) features of each graph:
        # In UPFD, the first node of each graph in the batch is the root node.
        # However, to be robust across batches, we find the first occurrence of each batch index.
        
        # This logic identifies the indices of the first node for each graph in the batch
        root_indices = (batch[1:] - batch[:-1]).nonzero(as_tuple=False).view(-1)
        root_indices = torch.cat([root_indices.new_zeros(1), root_indices + 1], dim=0)
        
        news_features = x[root_indices]
        return self.lin(news_features).relu()


In [ ]:
!mkdir -p src/models
%%writefile src/models/gnn_encoders.py
import torch
from torch_geometric.nn import GATConv, GCNConv, SAGEConv, global_max_pool

class GNNEncoder(torch.nn.Module):
    def __init__(self, model_type, in_channels, hidden_channels):
        super().__init__()
        
        if model_type == 'GCN':
            self.conv = GCNConv(in_channels, hidden_channels)
        elif model_type == 'SAGE':
            self.conv = SAGEConv(in_channels, hidden_channels)
        elif model_type == 'GAT':
            self.conv = GATConv(in_channels, hidden_channels)
        else:
            raise ValueError(f"Unsupported GNN model type: {model_type}")

    def forward(self, x, edge_index, batch):
        h = self.conv(x, edge_index).relu()
        h = global_max_pool(h, batch)
        return h


In [ ]:
!mkdir -p src/models
%%writefile src/models/hgfnd.py
import torch
import torch.nn as nn
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import scatter

class PropagationEncoder(nn.Module):
    r"""
    Encodes the propagation tree of each news piece.
    xi = ROOT(GNN(Pi))
    We concatenate the encoded root representation with the original news content features
    to generate an enhanced news representation v0 = f(sigma(xi_gnn_root \oplus xi_orig_root)).
    """
    def __init__(self, gnn_type: str, in_channels: int, hidden_channels: int):
        super().__init__()
        if gnn_type == 'SAGE':
            self.conv = SAGEConv(in_channels, hidden_channels)
        else:
            # Fallback/support for other types if needed, SAGE is default in HGFND
            from torch_geometric.nn import GCNConv, GATConv
            if gnn_type == 'GCN':
                self.conv = GCNConv(in_channels, hidden_channels)
            elif gnn_type == 'GAT':
                self.conv = GATConv(in_channels, hidden_channels)
            else:
                raise ValueError(f"Unsupported GNN type for PropagationEncoder: {gnn_type}")
                
        self.fc = nn.Linear(hidden_channels + in_channels, hidden_channels)

    def forward(self, x, edge_index, batch):
        # Run GNN over the propagation tree
        h_all = self.conv(x, edge_index).relu()
        
        # In UPFD, the first node of each graph in the batch is the root node.
        # Find the root node indices for the batch
        root_indices = (batch[1:] - batch[:-1]).nonzero(as_tuple=False).view(-1)
        root_indices = torch.cat([root_indices.new_zeros(1), root_indices + 1], dim=0)
        
        h_gnn_root = h_all[root_indices] # [num_graphs, hidden_channels]
        h_orig_root = x[root_indices]     # [num_graphs, in_channels]
        
        # Concatenate GNN root representation and original root news content
        h_concat = torch.cat([h_gnn_root, h_orig_root], dim=-1) # [num_graphs, hidden_channels + in_channels]
        
        # Linear projection with activation
        v0 = self.fc(h_concat).relu() # [num_graphs, hidden_channels]
        return v0

class HyperGATLayer(nn.Module):
    """
    Implements a single layer of Hypergraph Attention Network with a dual-level attention mechanism.
    1. Node-level attention: aggregates node representations to form hyperedge representations.
    2. Hyperedge-level attention: aggregates hyperedge representations to form node representations.
    """
    def __init__(self, hidden_channels: int):
        super().__init__()
        self.d = hidden_channels
        self.W1 = nn.Linear(self.d, self.d, bias=False)
        self.W2 = nn.Linear(self.d, self.d, bias=False)
        self.a1 = nn.Parameter(torch.randn(self.d, 1))
        self.a2 = nn.Parameter(torch.randn(2 * self.d, 1))
        self.leaky_relu = nn.LeakyReLU(0.2)
        
        # Initialize parameters
        nn.init.xavier_uniform_(self.a1)
        nn.init.xavier_uniform_(self.a2)

    def forward(self, v, H):
        """
        v: Node representations of shape [N, d]
        H: Incidence matrix of shape [N, M] (where H_ij = 1 if node i belongs to hyperedge j)
        """
        N, M = H.shape
        
        # --- 1. Node-level Attention for Hyperedge Representation ---
        # Equation (3): el_j = \sigma( \sum_{vk \in ej} \alpha_{jk} W1 vl-1_k )
        W1_v = self.W1(v) # [N, d]
        h_node = self.leaky_relu(W1_v) # [N, d]
        
        # attn_scores: exp(a1^T hk) of shape [N, 1]
        attn_scores = torch.matmul(h_node, self.a1) # [N, 1]
        exp_attn = torch.exp(attn_scores - attn_scores.max()) # [N, 1]
        
        # Sum exp_attn for each hyperedge: denominator = H_T * exp_attn [M, 1]
        H_T = H.t() # [M, N]
        denom = torch.matmul(H_T, exp_attn) + 1e-9 # [M, 1]
        
        # alpha_jk = H_T[j, k] * exp_attn[k] / denom[j]
        # alpha is of shape [M, N]
        alpha = H_T * exp_attn.view(1, -1) / denom # [M, N]
        
        # el_j = \sigma( \sum_k \alpha_jk W1_v[k] )
        # e = \sigma( alpha * W1_v ) of shape [M, d]
        e = torch.matmul(alpha, W1_v).relu() # [M, d]
        
        # --- 2. Hyperedge-level Attention for Node Representation ---
        # Equation (5): vl_i = \sigma( \sum_{ej \in Ei} \beta_{ij} W2 el_j )
        # rj = LeakyReLU([W2 el_j \oplus W1 vl-1_i])
        W2_e = self.W2(e) # [M, d]
        
        # Find connection indices where H[i, j] == 1
        i_idx, j_idx = H.nonzero(as_tuple=True)
        
        # Connected pair representations
        feat_i = W1_v[i_idx] # [num_edges, d]
        feat_j = W2_e[j_idx] # [num_edges, d]
        
        # Concatenate and apply activation
        concat_feat = torch.cat([feat_j, feat_i], dim=-1) # [num_edges, 2d]
        r_val = self.leaky_relu(concat_feat) # [num_edges, 2d]
        
        # Calculate attention scores
        scores = torch.matmul(r_val, self.a2).squeeze(-1) # [num_edges]
        
        # Compute softmax over j for each node i
        max_score = scatter(scores, i_idx, dim=0, dim_size=N, reduce='max')
        scores_exp = torch.exp(scores - max_score[i_idx])
        sum_exp = scatter(scores_exp, i_idx, dim=0, dim_size=N, reduce='sum') + 1e-9
        beta = scores_exp / sum_exp[i_idx] # [num_edges]
        
        # Compute new node representation
        weighted_e = beta.view(-1, 1) * W2_e[j_idx] # [num_edges, d]
        v_next = scatter(weighted_e, i_idx, dim=0, dim_size=N, reduce='sum').relu() # [N, d]
        
        return v_next

class HGFND(nn.Module):
    """
    HGFND: Hypergraph Neural Network for Fake News Detection.
    Connects news pieces through hyperedges and processes their relations transductively.
    """
    def __init__(self, config: dict, in_channels: int, out_channels: int):
        super().__init__()
        model_cfg = config['model']
        self.gnn_type = model_cfg.get('gnn_type', 'SAGE')
        self.hidden_channels = model_cfg.get('hidden_channels', 128)
        self.num_layers = model_cfg.get('hgfnd_layers', 2)
        
        # 1. Propagation Encoder to initialize node representations v0
        self.prop_encoder = PropagationEncoder(self.gnn_type, in_channels, self.hidden_channels)
        
        # 2. HyperGAT Layers
        self.layers = nn.ModuleList([
            HyperGATLayer(self.hidden_channels) for _ in range(self.num_layers)
        ])
        
        # 3. Classifier Head
        self.classifier = nn.Linear(self.hidden_channels, out_channels)
        
    def forward(self, x, edge_index, batch, H):
        """
        x: All node features across all propagation trees merged together
        edge_index: Edge index of all trees
        batch: Graph assignment of all nodes
        H: Global incidence matrix [N, M]
        """
        # Step 1: Encode propagation trees to get initial node representations v0
        v = self.prop_encoder(x, edge_index, batch) # [N, hidden_channels]
        
        # Step 2: Pass through HyperGAT layers
        for layer in self.layers:
            v = layer(v, H)
            
        # Step 3: Classify
        logits = self.classifier(v)
        return logits.log_softmax(dim=-1)


In [ ]:
!mkdir -p src/models
%%writefile src/models/upfd_model.py
import torch
import torch.nn as nn
from src.models.gnn_encoders import GNNEncoder
from src.models.text_encoders import TextEncoder
from src.models.classifier import MLPClassifier
from src.models.cmcg import MultiHeadCoAttentionLayer

class SentimentEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        # assuming sentiment feature is provided separately (e.g., shape [batch_size, in_channels])
        self.lin = nn.Linear(in_channels, hidden_channels)

    def forward(self, sentiment_features):
        return self.lin(sentiment_features).relu()

class UPFDModel(nn.Module):
    def __init__(self, config, in_channels, out_channels):
        super().__init__()
        model_cfg = config['model']
        self.use_hgfnd = model_cfg.get('use_hgfnd', False)
        
        if self.use_hgfnd:
            from src.models.hgfnd import HGFND
            self.hgfnd_model = HGFND(config, in_channels, out_channels)
            return

        self.use_gnn = model_cfg.get('use_gnn', True)
        self.use_text = model_cfg.get('use_text', True)
        self.use_sentiment = model_cfg.get('use_sentiment', False)
        self.use_cmcg = model_cfg.get('use_cmcg', False)
        
        hidden_channels = model_cfg['hidden_channels']

        self.gnn_encoder = None
        self.text_encoder = None
        self.sentiment_encoder = None
        combined_channels = 0

        if self.use_gnn:
            self.gnn_encoder = GNNEncoder(
                model_cfg['gnn_type'], in_channels, hidden_channels
            )
            combined_channels += hidden_channels

        if self.use_text:
            self.text_encoder = TextEncoder(in_channels, hidden_channels)
            combined_channels += hidden_channels

        if self.use_cmcg and self.use_gnn and self.use_text:
            self.co_attention = MultiHeadCoAttentionLayer(hidden_channels, hidden_channels, num_heads=2)
            # Channels remain the same after co-attention since we still concat them

        if self.use_sentiment:
            sentiment_dim = model_cfg.get('sentiment_dim', 1) # default to 1D sentiment score
            self.sentiment_encoder = SentimentEncoder(sentiment_dim, hidden_channels)
            combined_channels += hidden_channels

        if combined_channels == 0:
            raise ValueError("At least one encoder (GNN, Text, or Sentiment) must be enabled in the config.")

        self.classifier = MLPClassifier(combined_channels, out_channels)

    def forward(self, x, edge_index, batch, sentiment_features=None, H=None):
        if self.use_hgfnd:
            return self.hgfnd_model(x, edge_index, batch, H)

        embeddings = []
        h_gnn = None
        h_text = None

        if self.use_gnn:
            h_gnn = self.gnn_encoder(x, edge_index, batch)
            
        if self.use_text:
            h_text = self.text_encoder(x, batch)

        if self.use_cmcg and h_gnn is not None and h_text is not None:
            h_gnn, h_text = self.co_attention(h_gnn, h_text)
            
        if h_gnn is not None:
            embeddings.append(h_gnn)
        if h_text is not None:
            embeddings.append(h_text)

        if self.use_sentiment:
            if sentiment_features is None:
                # Fallback to zeros if not provided in the batch
                sentiment_features = torch.zeros(batch.max().item() + 1, self.sentiment_encoder.lin.in_features).to(x.device)
            h_sent = self.sentiment_encoder(sentiment_features)
            embeddings.append(h_sent)

        # Concatenate embeddings from all active encoders
        if len(embeddings) > 1:
            h = torch.cat(embeddings, dim=-1)
        else:
            h = embeddings[0]

        return self.classifier(h)



In [ ]:
%%writefile main.py
import torch
import os.path as osp
from torch_geometric.datasets import UPFD
from torch_geometric.transforms import ToUndirected
from torch_geometric.data import Batch
from src.utils import load_config, get_device
from src.data import get_loaders
from src.models.upfd_model import UPFDModel
from src.trainer import Trainer, TransductiveTrainer

def build_hypergraph(train_dataset, val_dataset, test_dataset, config):
    """
    Builds the global hypergraph incidence matrix H representing news relations.
    1. User Hyperedges: Shared user features across graphs.
    2. Time Hyperedges: Round relative creation time to proximal bins.
    3. Entity Hyperedges: K-Means clustering of news content.
    """
    import numpy as np
    from sklearn.cluster import KMeans
    
    all_graphs = list(train_dataset) + list(val_dataset) + list(test_dataset)
    N = len(all_graphs)
    print(f"Building hypergraph with N={N} nodes (graphs)...")
    
    # 1. User Hyperedges
    user_to_graphs = {}
    for g_idx, data in enumerate(all_graphs):
        # Exclude the root node (news content node) from user matching
        user_features = data.x[1:]
        for i in range(user_features.size(0)):
            feat = tuple(np.round(user_features[i].numpy(), 6))
            if feat not in user_to_graphs:
                user_to_graphs[feat] = set()
            user_to_graphs[feat].add(g_idx)
            
    # Filter users to build hyperedges (keep users who shared at least TWO graphs)
    user_hyperedges = [list(graphs) for graphs in user_to_graphs.values() if len(graphs) >= 2]
    print(f"Constructed {len(user_hyperedges)} user-based hyperedges (size >= 2).")
    
    # 2. Time Hyperedges
    # Load profile features for temporal information to ensure feature-agnostic robust creation
    path = osp.join(osp.dirname(osp.realpath(__file__)), config['data']['data_dir'])
    p_train = UPFD(path, config['data']['dataset'], 'profile', 'train')
    p_val = UPFD(path, config['data']['dataset'], 'profile', 'val')
    p_test = UPFD(path, config['data']['dataset'], 'profile', 'test')
    all_p_graphs = list(p_train) + list(p_val) + list(p_test)
    
    time_decimals = config['model'].get('time_decimals', 2)
    time_to_graphs = {}
    for g_idx, p_data in enumerate(all_p_graphs):
        if p_data.x.size(1) > 9:
            time_vals = p_data.x[1:, 9].numpy()
            for t in time_vals:
                t_rounded = np.round(t, time_decimals)
                if t_rounded not in time_to_graphs:
                    time_to_graphs[t_rounded] = set()
                time_to_graphs[t_rounded].add(g_idx)
                
    time_hyperedges = [list(graphs) for graphs in time_to_graphs.values() if len(graphs) >= 2]
    print(f"Constructed {len(time_hyperedges)} time-based hyperedges (size >= 2).")
    
    # 3. Entity Hyperedges
    root_features = []
    for data in all_graphs:
        root_features.append(data.x[0].numpy())
    root_features = np.vstack(root_features)
    
    num_clusters = config['model'].get('entity_clusters', 50)
    kmeans = KMeans(n_clusters=min(num_clusters, N), random_state=42, n_init='auto')
    cluster_labels = kmeans.fit_predict(root_features)
    
    entity_to_graphs = {}
    for g_idx, label in enumerate(cluster_labels):
        if label not in entity_to_graphs:
            entity_to_graphs[label] = set()
        entity_to_graphs[label].add(g_idx)
        
    entity_hyperedges = [list(graphs) for graphs in entity_to_graphs.values() if len(graphs) >= 2]
    print(f"Constructed {len(entity_hyperedges)} entity-based hyperedges (size >= 2).")
    
    # Combine hyperedges
    all_hyperedges = user_hyperedges + time_hyperedges + entity_hyperedges
    M = len(all_hyperedges)
    
    H = torch.zeros((N, M), dtype=torch.float)
    for h_idx, graphs in enumerate(all_hyperedges):
        for g_idx in graphs:
            H[g_idx, h_idx] = 1.0
            
    print(f"Global incidence matrix shape: {H.shape}")
    return H

def run_experiment(config):
    # Get device
    device = get_device(config['training']['device'])
    print(f"Using device: {device}")

    # Prepare data
    train_loader, val_loader, test_loader, train_dataset = get_loaders(config)
    
    model_cfg = config['model']
    use_hgfnd = model_cfg.get('use_hgfnd', False)
    
    if use_hgfnd:
        print("Initializing HGFND transductive experiment...")
        path = osp.join(osp.dirname(osp.realpath(__file__)), config['data']['data_dir'])
        val_dataset = UPFD(path, config['data']['dataset'], config['data']['feature'], 'val', ToUndirected())
        test_dataset = UPFD(path, config['data']['dataset'], config['data']['feature'], 'test', ToUndirected())
        
        all_graphs = list(train_dataset) + list(val_dataset) + list(test_dataset)
        
        # Build global batch & incidence matrix H
        H = build_hypergraph(train_dataset, val_dataset, test_dataset, config)
        giant_batch = Batch.from_data_list(all_graphs)
        
        # Prepare transductive split indices and labels
        N_train = len(train_dataset)
        N_val = len(val_dataset)
        N_all = len(all_graphs)
        
        train_idx = torch.arange(0, N_train, dtype=torch.long)
        val_idx = torch.arange(N_train, N_train + N_val, dtype=torch.long)
        test_idx = torch.arange(N_train + N_val, N_all, dtype=torch.long)
        
        y = torch.cat([data.y for data in all_graphs], dim=0)
        
        # Initialize model
        model = UPFDModel(
            config=config,
            in_channels=train_dataset.num_features,
            out_channels=train_dataset.num_classes
        )
        
        # Optimizer
        optimizer = torch.optim.Adam(
            model.parameters(), 
            lr=config['training']['lr'], 
            weight_decay=config['training']['weight_decay']
        )
        
        # Transductive Trainer
        trainer = TransductiveTrainer(
            model=model,
            optimizer=optimizer,
            device=device,
            giant_batch=giant_batch,
            H=H,
            train_idx=train_idx,
            val_idx=val_idx,
            test_idx=test_idx,
            y=y
        )
        
        # Start training
        trainer.fit(config['training']['epochs'])
        test_acc = trainer.test(trainer.test_idx)
        return test_acc
    else:
        # Initialize standard GNN/Text model
        model = UPFDModel(
            config=config,
            in_channels=train_dataset.num_features,
            out_channels=train_dataset.num_classes
        )
        
        # Optimizer
        optimizer = torch.optim.Adam(
            model.parameters(), 
            lr=config['training']['lr'], 
            weight_decay=config['training']['weight_decay']
        )
        
        # Trainer
        trainer = Trainer(
            model=model,
            optimizer=optimizer,
            device=device,
            train_loader=train_loader,
            val_loader=val_loader,
            test_loader=test_loader
        )
        
        # Start training
        trainer.fit(config['training']['epochs'])
        test_acc = trainer.test(trainer.test_loader)
        return test_acc

def main():
    # Load configuration
    config = load_config('config.yaml')
    run_experiment(config)

if __name__ == "__main__":
    main()



In [ ]:
%%writefile config.yaml
# UPFD Project Configuration

data:
  dataset: "gossipcop" # choices: ['politifact', 'gossipcop']
  feature: "spacy"      # choices: ['profile', 'spacy', 'bert', 'content']
  batch_size: 128
  data_dir: "dataset"

model:
  gnn_type: "SAGE"      # choices: ['GCN', 'GAT', 'SAGE']
  hidden_channels: 128
  use_gnn: true
  use_text: true
  # Future extensions
  use_sentiment: false
  # HGFND settings
  use_hgfnd: false      # Set to true to use the HGFND architecture from the paper
  hgfnd_layers: 2
  entity_clusters: 50
  time_decimals: 2


training:
  lr: 0.001
  weight_decay: 0.01
  epochs: 60
  device: "auto" # 'auto', 'cuda', or 'cpu'




In [ ]:
%%writefile colab_experiments.py
import itertools
import pandas as pd
import time
import torch
import random
import numpy as np
import os

# Ensure this script is run in an environment where main.py can be imported
from main import run_experiment

# Colab-specific imports for Google Sheets
try:
    from google.colab import auth
    from google.auth import default
    import gspread
    COLAB_ENV = True
except ImportError:
    print("Not running in Google Colab, or gspread/auth not installed. Sheet logging will be skipped/simulated.")
    COLAB_ENV = False

def setup_google_sheet(sheet_name="UPFD_Experiments_HGFND"):
    if not COLAB_ENV:
        return None
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    
    try:
        sh = gc.open(sheet_name)
        worksheet = sh.sheet1
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(sheet_name)
        worksheet = sh.sheet1
        # Set up headers
        headers = ["Seed", "Dataset", "Feature", "Architecture Type", "GNN Type", "Use GNN", "Use Text", "Use HGFND (Hypergraph)", "Epochs", "Accuracy"]
        worksheet.append_row(headers)
    
    return worksheet

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def main():
    worksheet = setup_google_sheet("Fake_News_Detection_HGFND_Results")
    results_file = "fake_news_detection_results.csv"
    
    # Init CSV local file
    if not os.path.exists(results_file):
        df_init = pd.DataFrame(columns=[
            "Seed", "Dataset", "Feature", "Architecture Type", "GNN Type", 
            "Use GNN", "Use Text", "Use HGFND", "Epochs", "Accuracy"
        ])
        df_init.to_csv(results_file, index=False)
        print(f"Created local results file {results_file} for logging.")
    else:
        print(f"Appending to existing local results file {results_file}.")
        
    seeds = [42, 2026]
    datasets = ["gossipcop"]
    features = ["bert"] # Can add "spacy"
    
    # 10 rigorous experiments ONLY with GNN, Text-based models, and HGFND (hiperaristas)
    # Each configuration: (architecture_name, gnn_type, use_gnn, use_text, use_hgfnd)
    experiments_matrix = [
        # 1. Text-Only baseline
        ("Text-Only", "SAGE", False, True, False),
        
        # 2-4. GNN-Only baselines (propagation trees only)
        ("GNN-Only (GCN)", "GCN", True, False, False),
        ("GNN-Only (SAGE)", "SAGE", True, False, False),
        ("GNN-Only (GAT)", "GAT", True, False, False),
        
        # 5-7. GNN + Text baselines (Early Fusion / Concatenation)
        ("GNN+Text (GCN)", "GCN", True, True, False),
        ("GNN+Text (SAGE)", "SAGE", True, True, False),
        ("GNN+Text (GAT)", "GAT", True, True, False),
        
        # 8-10. HGFND (Hypergraph Neural Network)
        ("HGFND (GCN)", "GCN", True, True, True),
        ("HGFND (SAGE)", "SAGE", True, True, True),
        ("HGFND (GAT)", "GAT", True, True, True)
    ]

    for seed in seeds:
        for dataset in datasets:
            for feature in features:
                for arch_name, gnn_type, use_gnn, use_text, use_hgfnd in experiments_matrix:
                    set_seed(seed)
                    
                    config = {
                        "data": {
                            "dataset": dataset,
                            "feature": feature,
                            "batch_size": 128,
                            "data_dir": "dataset"
                        },
                        "model": {
                            "gnn_type": gnn_type,
                            "hidden_channels": 128,
                            "use_gnn": use_gnn,
                            "use_text": use_text,
                            "use_sentiment": False,
                            "use_cmcg": False,
                            # HGFND parameters
                            "use_hgfnd": use_hgfnd,
                            "hgfnd_layers": 2,
                            "entity_clusters": 50,
                            "time_decimals": 2
                        },
                        "training": {
                            "lr": 0.001,
                            "weight_decay": 0.01,
                            "epochs": 30, # Optimized epoch length for comparative evaluation
                            "device": "auto"
                        }
                    }
                    
                    print(f"\n--- Running Experiment: {arch_name} ---")
                    print(f"Seed: {seed}, Dataset: {dataset}, Feature: {feature}")
                    
                    try:
                        acc = run_experiment(config)
                        acc_val = float(acc)
                        print(f"Achieved Accuracy: {acc_val:.4f}")
                        
                        # Log to CSV
                        row_df = pd.DataFrame([{
                            "Seed": seed, "Dataset": dataset, "Feature": feature, 
                            "Architecture Type": arch_name, "GNN Type": gnn_type, 
                            "Use GNN": use_gnn, "Use Text": use_text, "Use HGFND": use_hgfnd, 
                            "Epochs": config['training']['epochs'], "Accuracy": acc_val
                        }])
                        row_df.to_csv(results_file, mode='a', header=False, index=False)
                        
                        # Log to Google Sheets
                        if worksheet is not None:
                            row = [
                                seed, dataset, feature, arch_name, gnn_type,
                                use_gnn, use_text, use_hgfnd, 
                                config['training']['epochs'], acc_val
                            ]
                            worksheet.append_row(row)
                            time.sleep(1)
                            
                    except Exception as e:
                        print(f"Experiment failed: {e}")
                        row_df = pd.DataFrame([{
                            "Seed": seed, "Dataset": dataset, "Feature": feature, 
                            "Architecture Type": arch_name, "GNN Type": gnn_type, 
                            "Use GNN": use_gnn, "Use Text": use_text, "Use HGFND": use_hgfnd, 
                            "Epochs": config['training']['epochs'], "Accuracy": f"ERROR: {str(e)}"
                        }])
                        row_df.to_csv(results_file, mode='a', header=False, index=False)
                        
                        if worksheet is not None:
                            row = [
                                seed, dataset, feature, arch_name, gnn_type,
                                use_gnn, use_text, use_hgfnd, 
                                config['training']['epochs'], f"ERROR: {str(e)}"
                            ]
                            worksheet.append_row(row)
                            time.sleep(1)

if __name__ == "__main__":
    main()


# 1. Install Dependencies

In [ ]:
!pip install torch_geometric PyYAML gspread

# 2. Download and Extract Dataset
Fixes the 404 error from PyTorch Geometric.

In [ ]:
!mkdir -p dataset/gossipcop/raw
!curl -L -o dataset/gossipcop/raw/data.zip "https://data.pyg.org/datasets/upfd_gossipcop.zip"
!cd dataset/gossipcop/raw && unzip -o data.zip && rm data.zip

# 3. Run Experiments
This will authenticate with Google Sheets and start the training loops.

In [ ]:
!python colab_experiments.py